# Command centerThe workspace for this project: setup, every knob, the features being built, and the outputyou check them by. Runs top to bottom on a fresh runtime — no hidden state, no cell that hasto be run out of order.**This is the template's own notebook.** What this project is *for* gets written when the repois cloned — see `STATE.md` §1. Until then the config cell carries only the paths every projectneeds, and the pipeline section at the bottom is deliberately empty.How it's used: features get built *here*, in as many cells as it takes to see them working,and move into `src/` once they've stopped changing. `AGENTS.md` §6 has that lifecycle;`PLAYBOOK.md` has the one-time setup and the Colab ↔ GitHub round trip.

## 1. SetupRun once per fresh runtime. Re-run after **Runtime → Restart**.

In [ ]:
"""Prepare the runtime: mount Drive, fetch the repo, install what Colab doesn't ship.Nothing in this cell is a choice you make. The mount point is a Colab constant, the repo iswherever this project lives, and the package list is simply whatever the code imports — thethings you actually tune live in the config cell below.Opening this notebook from GitHub gives you the notebook and nothing else, so the repo iscloned separately to put `src/` on the import path. That is what makes `from pipeline import ...`work in the cells further down.Idempotent: re-run it after **Runtime > Restart**. An existing clone is pulled, not re-cloned."""import subprocessimport sysfrom pathlib import Path# ON CLONE: point this at your own repo.REPO_URL = "https://github.com/vak-sah/NDD-notebook-driven-development.git"REPO_DIR = Path("/content/repo")from google.colab import drivedrive.mount("/content/drive")if REPO_DIR.exists():    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=True)else:    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)if str(REPO_DIR / "src") not in sys.path:    sys.path.insert(0, str(REPO_DIR / "src"))# Packages Colab doesn't already have. Pin versions — a runtime six months from now should# resolve to the same thing this one did. Empty until the project needs something.PACKAGES: list[str] = [    # "some-lib==1.2.3",]if PACKAGES:    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES], check=True)print(f"Drive mounted \u00b7 repo at {REPO_DIR} \u00b7 {len(PACKAGES)} extra package(s)")

## 2. ConfigEvery knob in the project, in one cell. Edit here, then run everything below.

In [ ]:
"""Every knob this project has, and why each one is set the way it is.Read this cell on its own and you should come away knowing what you can change, what thealternatives were, and why the current value won — that last part is what gets forgotten first.Values are grouped by the decision you're making, not by whichever module happens to consumethem. Code downstream takes these as arguments; nothing reads a config module behind your back."""from pathlib import Path# --- Where files live --------------------------------------------------------------------# Everything that isn't code, config or docs: data, weights, caches, outputs, secrets.# None of it belongs in git — .gitignore keeps it out.## DRIVE_ROOT — alternatives considered:#   /content/drive/MyDrive/<project>  <- current. survives runtime teardown, one place to#                                        look, and it's the path recorded in PLAYBOOK.md#   /content/<anything>                  the runtime's own disk. faster, but wiped every time#                                        the runtime recycles, which means lost work#   a folder inside the repo             would put data in git. no## ON CLONE: rename the last folder after your project. Two clones both left on "NDD" would# share one Drive folder and quietly overwrite each other's data. Change it in PLAYBOOK.md# too — those are the only two places this path is allowed to appear.DRIVE_ROOT = Path("/content/drive/MyDrive/NDD")DATA_DIR = DRIVE_ROOT / "data"          # inputs, placed here by hand (PLAYBOOK.md)OUTPUTS_DIR = DRIVE_ROOT / "outputs"    # anything a run producesfor _d in (DATA_DIR, OUTPUTS_DIR):    _d.mkdir(parents=True, exist_ok=True)# --- Project knobs -----------------------------------------------------------------------# Empty until this clone has features of its own. Each one arrives as its own block: the# value, the alternatives that were weighed, and a line on why the current value won.print(f"DRIVE_ROOT   {DRIVE_ROOT}")print(f"DATA_DIR     {DATA_DIR}")print(f"OUTPUTS_DIR  {OUTPUTS_DIR}")

## 3. PipelineThe end-to-end stub: read a file, pass the records through untouched, write them out. It provesnotebook -> `src/` -> `tests/` -> CI is connected, and gives the first real feature something toreplace. `src/pipeline/stub.py` owns it; delete it once a real first stage exists.

In [ ]:
"""Run the stub end to end and show the result.Writes a small input file into DATA_DIR, runs the pipeline, prints what came back out. Replacethis cell's call when the stub is replaced by a real first stage."""from pipeline import stubin_path = DATA_DIR / "stub_input.txt"out_path = OUTPUTS_DIR / "stub_output.txt"in_path.write_text("alpha\nbeta\ngamma\n")written = stub.run(in_path, out_path)print(f"{written} record(s) -> {out_path}\n")print(out_path.read_text())